# Programmatic access to the MICrONS dataset using CAVE and NGLUI

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllenInstitute/connectomics_at_cosyne/blob/main/examples/Tutorial_NGLUI.ipynb)


This tutorial walks through the key functions needed to access the MICrONS dataset programmatically and highlights key resources within it. This tutorial is written for the MICrONS dataset specifically, but note that the underlying technology (CAVE) is being used for multiple connectomics dataset.

This tutorial is designed to be run in Google Colab. Some adjustments should be made if run locally, mostly around installation and authentication — see [CAVEclient documentation](https://caveconnectome.github.io/CAVEclient/) for more information.

**The CAVEclient** is a python library that facilitates communication with a CAVE system. This package can can be installed with pip.


*Note:* when using colab, if you use uv to manage the package installs, restarting the kernel and reinstalling the packages will be much faster

In [ ]:
!uv pip install caveclient
!uv pip install nglui[full]

In [ ]:
# import python modules
from os.path import join as pjoin
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Initialize CAVEclient with a datastack

Datasets in CAVE are organized as datastacks. These are a combination of an EM dataset, a segmentation and a set of annotations. The datastack for MICrONS public release is `minnie65_public`.

When you instantiate your client with this datastack, it loads all relevant information to access it.

**If you have already saved your CAVE TOKEN to secrets, google will load this for you.** You may be asked to allow the notebook access to your secrets.

In [ ]:
from google.colab import userdata
import caveclient
client = caveclient.CAVEclient('minnie65_public',
                               auth_token=userdata.get('CAVE_TOKEN'))

## Neuroglancer User Interface (NGLui)
[__NGLui__](https://caveconnectome.github.io/nglui/) is a library for programmatically generating shareable Neuroglancer states in order to explore and visualize large, 3d image datasets. It is designed to be used in conjunction with the Neuroglancer web viewer and the CAVE analysis ecosystem. It aims to separate the rules you want to use to generate states from the data you want to visualize, allowing you make reusable code to generate states from different datasets and analyses.

In this section we use the `statebuilder` module to generate neuroglancer links, or specifically their underlying JSON 'states'.

In [ ]:
from nglui import statebuilder

# setting the default to generate spelunker links
statebuilder.set_default_neuroglancer_site('spelunker')

# Example: thalamic axon
example_cell_id = 864691136489225746
example_cell_id = client.chunkedgraph.suggest_latest_roots(example_cell_id, timestamp = client.materialize.get_timestamp()) # get current version from an old root

statebuilder.helpers.make_neuron_neuroglancer_link(
    client,
    example_cell_id,
    show_inputs=False,
    show_outputs=True,
)

These 'helper' functions will do a lot of the work of rendering states for you, but sometimes you want more granular access to different parts of the state.

We will step through:
 * specifying the imagery and segmentation layers
 * adding a synapse annotation layer, linked to the segmentation
 * adding an extra segmentation layer
 * adding annotation tags and description to the neuroglancer state

 Extensive documentation and many more complete examples can be found on the [NGLui Statebuilder Documentation page](https://caveconnectome.github.io/nglui/usage/statebuilder/)

### Statebuilder with user-defined layers

A neuroglancer state is made of multiple aligned 'layers' that will be one of:
* `imagery`
* `segmentation`
* `annotation`

And there are specific modules and functions for generating each of these. The final step is to wrap all these layers and necessary information together and `render_state` to make the neuroglancer output

In [ ]:
from nglui.statebuilder import ViewerState, ImageLayer, SegmentationLayer

# Load the correct paths from the CAVEclient
ViewerState().add_layers_from_client(client).layers

In [ ]:
# Initialize viewer state and add layers (basic view, no segments, all defaults)
viewer_state = (
    ViewerState()
    .add_layers_from_client(client)  # add default imagery and segmentation
)

viewer_state.to_link()


In [ ]:
dir(ViewerState)
ViewerState.set_viewer_properties?

In [ ]:
# Granular control over ever layer:

img_layer = ImageLayer(
    source='precomputed://https://bossdb-open-data.s3.amazonaws.com/iarpa_microns/minnie/minnie65/em',
)

seg_layer = (
    SegmentationLayer()
    .add_source('graphene://https://minnie.microns-daf.com/segmentation/table/minnie3_v1')
    .add_segments([example_cell_id]) # add example cell
    .add_segment_colors(
        {
            example_cell_id: '#FFFFFF',
        }
    )
)
seg_layer.alpha_3d = 0.8 # set the opacity of selected segments

# Package that together:
viewer_state = (
    ViewerState()
    .add_layer(img_layer)
    .add_layer(seg_layer)
    .set_viewer_properties(layout='3d',
                           dimensions = [8,8,40],
                           scale_3d=2e5
                           )
)

# display url link
viewer_state.to_link()

In [ ]:
# Show the internals of the neuroglancer state
viewer_state.to_dict()

The dictionary output is useful for manually changing things in the state, but beware: it is very easy to break things when altering the state directly. It is better to use the helper functions and arguments where possible.

Usually, what you will want is a neuroglancer link directly:

In [ ]:
# now render the state as a long url
viewer_state.to_url()

But this long-form url is unwieldy to work with. The more annotations and layers you add, the longer the url gets. Generally, you will want one of two things:

* upload the long url to a 'link shortener' that gives you a much simpler url
* create a clickable html link that hides the full url from you

In [ ]:
# Generate shortened url
print(viewer_state.to_link_shortener(client))

print('  ')

# Generate a clickable html
viewer_state.to_link()

### Adding synapse information

The `make_neuron_neuroglancer_link` helper above can create a state with synapses, but you may want more granular control of which synapses to include and how they render. We can manage the same effect with a 'linked annotation layer'

We starte with querying the synapses directly with CAVEclient

In [ ]:
# Get synapse table (CAVE query)
syn_out = client.materialize.synapse_query(pre_ids=example_cell_id,
                                           split_positions=False,
                                           desired_resolution=[4,4,40], # needs to be this resolution for neuroglancer
                                           )

In [ ]:
syn_out

In [ ]:
from nglui.statebuilder import AnnotationLayer

annotation_layer = AnnotationLayer(
    name='synapses',
    linked_segmentation='seg', # Uses the `seg` name to link annotations to the segmentation layer
    color = '#FF0000', # set color of points
    )

annotation_layer.add_points(
    data=syn_out, # we tell statebuilder where to find the data we want
    point_column='ctr_pt_position', # which column to pull position info from
    segment_column= 'post_pt_root_id', # which column to link segmentation from (!can no longer add two linked segments)
    data_resolution=[4, 4, 40],
    )

In [ ]:
(
    ViewerState()
    .add_layer(img_layer)
    .add_layer(seg_layer)
    .add_layer(annotation_layer)
    .set_viewer_properties(layout='3d',
                           dimensions = [8,8,40],
                           scale_3d=2e5
                           )
).to_link()

### Add post-synaptic synaptic compartment information
The post-synaptic compartment (whether spine, shaft, or soma) is saved in a CAVE table `synapse_target_predictions_ssa_v2`, and can be accessed with CAVE query.

We can add this as a 'description' to the annotation points in the neuroglancer state.

We can also specify what tags to include, for example if manually labeling the cleft type as:

* disk
* horseshoe
* ring
* complex

In [ ]:
# CAVE query to find the postsynaptic compartment of our output synapses
postsynaptic_target = client.materialize.tables.synapse_target_predictions_ssa_v2(target_id=syn_out.id).query()

print(len(syn_out), len(postsynaptic_target))
postsynaptic_target.head()

In [ ]:
# now merge the columns from the postsynaptic targets with all synapses
# syn_out_target = pd.merge(syn_out, postsynaptic_target[['id','tag']], on='id', how='left').fillna('unknown').rename(columns={'tag':'target'})
syn_out_target = pd.merge(syn_out, postsynaptic_target[['id','tag']], on='id', how='inner').rename(columns={'tag':'target'})


annotation_layer = AnnotationLayer(
    name='synapses',
    linked_segmentation='seg', # Uses the `seg` name to link annotations to the segmentation layer
    color = '#FF0000', # set color of points
    tags=['disk','horseshoe','ring','complex'], # pregenerate the tags to use
    )

annotation_layer.add_points(
    data=syn_out_target, # we tell statebuilder where to find the data we want
    point_column='ctr_pt_position', # which column to pull position info from
    description_column='target', # adds the label from the syanptic compartment
    segment_column= 'post_pt_root_id', # which column to link segmentation from (!can no longer add two linked segments)
    data_resolution=[4, 4, 40],
    )


(
    ViewerState()
    .add_layer(img_layer)
    .add_layer(seg_layer)
    .add_layer(annotation_layer)
    .set_viewer_properties(layout='3d',
                           dimensions = [8,8,40],
                           scale_3d=2e5
                           )
).to_link()

### Add the cleft segmentation layer

The segmentation of the synaptic clefts is available as its own layer. Unfortunately, the clefts are not 'meshed' so you cannot see them in 3D, but you can in 2D.

Add the clefts: (all layer variables repeated here for convenience)

In [ ]:
cleft_seg_source = "precomputed://https://bossdb-open-data.s3.amazonaws.com/iarpa_microns/minnie/minnie65/clefts-sharded"

In [ ]:
cleft_layer = (
    SegmentationLayer(name='cleft')
    .add_source(cleft_seg_source)
)
cleft_layer.not_selected_alpha = 0.8 # set the opacity of unselected segments


# here we render as a dictionary, to edit something the statebuilder does not control (deactivating selection of a segmentation layer)
sb_dict = (
    ViewerState()
    .add_layer(img_layer)
    .add_layer(seg_layer)
    .add_layer(cleft_layer)
    .add_layer(annotation_layer)
    .set_viewer_properties(layout='4panel',
                           dimensions = [8,8,40],
                           scale_3d=2e3
                           )
).to_dict()

In [ ]:
# Edit the layer controls in the dictionary
sb_dict['layers'][2]['pick'] =  False
sb_dict['layers'][2]['hoverHighlight'] = False

In [ ]:
(ViewerState(base_state=sb_dict)
    .set_viewer_properties(layout='4panel',
                           dimensions = [8,8,40],
                           scale_3d=2e3
                           )
).to_link()

## Posting NGL states to the link shortener
In Neuroglancer, on the upper right, there is a 'Share' button.

When you click this link, you will be prompted for a google login. Use any google profile, but it is convenient to use the same as for this notebook.

This will upload the text of your neuroglancer state (the JSON format file, also accessed with the {} button) to a server, and returns a lookup number to access that state. The link with this state is automatically copied to your system's clipboard.

When you paste it, you will see something like this:

> https://ngl.microns-explorer.org/#!middleauth+https://global.daf-apis.com/nglstate/api/v1/6047968401555456

NLGUI can read the information directly out of that uploaded state with a tool called `Parser`:

In [ ]:
from nglui import parser

client = caveclient.CAVEclient('minnie65_public',
                               auth_token=userdata.get('CAVE_TOKEN'))
state_json = client.state.get_state_json(6047968401555456)
state = parser.StateParser(state_json)

You can now access different aspects of the state. For example, to get a list of all layers and their core info, you can use the `layer_dataframe` method.

In [ ]:
state.layer_dataframe()

This will give you a table with a row for each layer and columns for layer name, type, source, and whether the layer is archived (i.e. visible) or not.

With `parser` you can get a list of all annotations with the annotation_dataframe method.

In [ ]:
state.annotation_dataframe()

This will give you a dataframe where each row is an annotation, and columns show `layer name, `point` locations, annotation type, annotation id, descriptive text, linked segmentations, tags, etc.

If you have multiple annotation layers, each layer is specified by the `layer` column and all points across all layers are concatenated together.

The point coordinates are returned at the resolution of the neuroglancer view (default: 4x4x40 nm). You can change this resolution with argument `point_resolution` which will rescale the points to the requested resolution.

The `description` column populates with any text you have attached to the point.

If you are using tags, the `expand_tags=True` argument will create a column for every tag and assign a boolean value to the row based on whether the tag is present in the annotation.

Another option that is sometimes useful is `split_points=True`, which will create a separate column for each x, y, or z coordinate in the annotation.

In [ ]:
state.annotation_dataframe(expand_tags=True, split_points=True, point_resolution=[1,1,1])

For further documentation and examples, consider: [NGLui state parser](https://caveconnectome.github.io/nglui/usage/parser/)